# Xóa dữ liệu null, bỏ mấy sản phẩm không còn liên kết nữa

In [2]:
import os
import polars as pl
import pandas as pd

In [3]:
PATH = "/content/drive/MyDrive/Colab Notebooks/CTH001/AmazonData"

## 2014

In [4]:
!cp "{PATH}/2014/meta_Baby.json.gz" "meta_Baby.json.gz"

In [5]:
!gunzip meta_Baby.json.gz

In [7]:
import ast
import os
import json

def fast_normalize(input_path, output_path, chunk_size=1000):
    print(f"Bắt đầu chuẩn hóa: {input_path}")

    with open(input_path, 'r', encoding='utf-8') as f_in, \
         open(output_path, 'w', encoding='utf-8') as f_out:

        buffer = []
        for line in f_in:
            line = line.strip()
            if not line: continue

            try:
                # ast.literal_eval nhanh và chuẩn hơn eval cho bài toán này
                data_dict = ast.literal_eval(line)
                buffer.append(json.dumps(data_dict) + '\n')

                # Khi đủ 1000 dòng thì ghi xuống ổ cứng 1 lần
                if len(buffer) >= chunk_size:
                    f_out.writelines(buffer)
                    buffer = []
            except:
                continue

        # Ghi nốt phần còn lại trong buffer
        if buffer:
            f_out.writelines(buffer)

    print(f"Hoàn thành! Đã lưu tại: {output_path}")
    print(f"Kích thước file mới: {os.path.getsize(output_path) / 1024 / 1024:.2f} MB")

In [8]:
fast_normalize("/content/meta_Baby.json", "/content/meta_Baby.jsonl")

Bắt đầu chuẩn hóa: /content/meta_Baby.json
Hoàn thành! Đã lưu tại: /content/meta_Baby.jsonl
Kích thước file mới: 105.82 MB


In [9]:
df = pd.read_json("meta_Baby.jsonl", lines=True)

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 71317 entries, 0 to 71316
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   asin         71317 non-null  object 
 1   categories   71317 non-null  object 
 2   description  65642 non-null  object 
 3   title        71241 non-null  object 
 4   price        57741 non-null  float64
 5   imUrl        71243 non-null  object 
 6   brand        27858 non-null  object 
 7   related      58721 non-null  object 
 8   salesRank    36 non-null     object 
dtypes: float64(1), object(8)
memory usage: 4.9+ MB


In [14]:
df[(df["title"].isnull()) & (df["imUrl"].isnull() == False)].to_json("a.json")

## 2023

In [16]:
!ls "{PATH}/2023"

Baby_Products.jsonl   df_review_2023.parquet	processed_reviews
df_meta_2023.parquet  meta_Baby_Products.jsonl


### Meta

In [4]:
!cp "{PATH}/2023/meta_Baby_Products.jsonl" "meta_Baby_Products.jsonl"

In [4]:
df_meta_2023 = pd.read_json("meta_Baby_Products.jsonl", lines=True)

In [16]:
df_meta_2023.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 217724 entries, 0 to 217723
Data columns (total 16 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   main_category    199844 non-null  object 
 1   title            217724 non-null  object 
 2   average_rating   217724 non-null  float64
 3   rating_number    217724 non-null  int64  
 4   features         217724 non-null  object 
 5   description      217724 non-null  object 
 6   price            60256 non-null   float64
 7   images           217724 non-null  object 
 8   videos           217724 non-null  object 
 9   store            214722 non-null  object 
 10  categories       217724 non-null  object 
 11  details          217724 non-null  object 
 12  parent_asin      217724 non-null  object 
 13  bought_together  0 non-null       float64
 14  subtitle         0 non-null       float64
 15  author           1 non-null       object 
dtypes: float64(4), int64(1), object(11)
me

In [28]:
# Assuming df_meta_2023 is your existing pandas DataFrame
df_title_null = df_meta_2023[
    df_meta_2023['title'].isnull() |
    (df_meta_2023['title'].astype(str).str.strip() == '') |
    (~df_meta_2023['title'].astype(str).str.contains(r"[a-zA-Z]", na=False)) |
    (df_meta_2023['title'].astype(str).str.match(r"^[./\d\s]+$", na=False))
]

df_title_null.info()

<class 'pandas.core.frame.DataFrame'>
Index: 34 entries, 1863 to 211632
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   main_category    32 non-null     object 
 1   title            34 non-null     object 
 2   average_rating   34 non-null     float64
 3   rating_number    34 non-null     int64  
 4   features         34 non-null     object 
 5   description      34 non-null     object 
 6   price            2 non-null      float64
 7   images           34 non-null     object 
 8   videos           34 non-null     object 
 9   store            34 non-null     object 
 10  categories       34 non-null     object 
 11  details          34 non-null     object 
 12  parent_asin      34 non-null     object 
 13  bought_together  0 non-null      float64
 14  subtitle         0 non-null      float64
 15  author           0 non-null      object 
dtypes: float64(4), int64(1), object(11)
memory usage: 4.5+ KB


In [27]:
# Assuming df_meta_2023 is your existing pandas DataFrame
df_images_empty = df_meta_2023[
    df_meta_2023['images'].apply(lambda x: len(x) == 0)
]

df_images_empty.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10 entries, 13782 to 182109
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   main_category    10 non-null     object 
 1   title            10 non-null     object 
 2   average_rating   10 non-null     float64
 3   rating_number    10 non-null     int64  
 4   features         10 non-null     object 
 5   description      10 non-null     object 
 6   price            2 non-null      float64
 7   images           10 non-null     object 
 8   videos           10 non-null     object 
 9   store            10 non-null     object 
 10  categories       10 non-null     object 
 11  details          10 non-null     object 
 12  parent_asin      10 non-null     object 
 13  bought_together  0 non-null      float64
 14  subtitle         0 non-null      float64
 15  author           0 non-null      object 
dtypes: float64(4), int64(1), object(11)
memory usage: 1.3+ KB


In [29]:
df_meta_2023_cleaned = df_meta_2023[~
 (
     (df_meta_2023["parent_asin"].isin(df_title_null["parent_asin"]))
     | (df_meta_2023["parent_asin"].isin(df_images_empty["parent_asin"]))
 )]

df_meta_2023_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
Index: 217680 entries, 0 to 217723
Data columns (total 16 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   main_category    199802 non-null  object 
 1   title            217680 non-null  object 
 2   average_rating   217680 non-null  float64
 3   rating_number    217680 non-null  int64  
 4   features         217680 non-null  object 
 5   description      217680 non-null  object 
 6   price            60252 non-null   float64
 7   images           217680 non-null  object 
 8   videos           217680 non-null  object 
 9   store            214678 non-null  object 
 10  categories       217680 non-null  object 
 11  details          217680 non-null  object 
 12  parent_asin      217680 non-null  object 
 13  bought_together  0 non-null       float64
 14  subtitle         0 non-null       float64
 15  author           1 non-null       object 
dtypes: float64(4), int64(1), object(11)
memory 

In [31]:
df_meta_2023_cleaned.head(3)

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,Baby,"Chicco Viaro Travel System, Teak",4.6,125,"[Aluminum, Imported, Convenient one-hand quick...","[Product Description, For ultimate convenience...",NaN,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Viaro Demo Video', 'url': 'https:/...",Chicco,"[Baby Products, Strollers & Accessories, Strol...",{'Product Dimensions': '38 x 41.25 x 25.5 inch...,B01C4319LO,NaN,NaN,NaN
1,AMAZON FASHION,Kisbaby Four Layers Muslin Lightweight Unisex ...,5.0,2,"[95% Cotton, 4 Layer Muslin, Hand Wash in Cold...",[You can choose bigger size If you confuse abo...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Kisbaby,"[Baby Products, Nursery, Bedding, Blankets & S...","{'Material': 'Muslin', 'Color': 'Blue Star', '...",B07FM4MJJP,NaN,NaN,NaN
2,Baby,EZTOTZ Meals with Milton - USA Made Toddler & ...,4.4,37,[WHAT IS MILTON?: Milton is the fun way for yo...,[],22.99,[{'thumb': 'https://m.media-amazon.com/images/...,[],EZTOTZ,[],{'Package Dimensions': '6.3 x 5 x 4.76 inches'...,B08WCG372G,NaN,NaN,NaN


In [32]:
df_meta_2023_cleaned.to_json('df_meta_2023_cleaned.jsonl', orient='records', lines=True)
print("DataFrame df_meta_2023_cleaned đã được lưu vào 'df_meta_2023_cleaned.jsonl'")

DataFrame df_meta_2023_cleaned đã được lưu vào 'df_meta_2023_cleaned.jsonl'


In [33]:
!cp "df_meta_2023_cleaned.jsonl" "{PATH}/2023/df_meta_2023_cleaned.jsonl"

In [42]:
# Chỉ quét cấu trúc file
lazy_df_meta_2023 = pl.scan_parquet("df_meta_2023.parquet")

lazy_df_meta_2023.collect_schema().names()

['main_category',
 'title',
 'average_rating',
 'rating_number',
 'features',
 'description',
 'price',
 'images',
 'videos',
 'store',
 'categories',
 'details',
 'parent_asin',
 'bought_together',
 'subtitle',
 'author',
 'df_index']

### Review

In [7]:
!cp "{PATH}/2023/Baby_Products.jsonl" "Baby_Products.jsonl"

In [4]:
!ls -lh

total 5.9G
-rw-r--r-- 1 root root  70K Mar 30 01:19 a.csv
-rw-r--r-- 1 root root    0 Mar 30 01:34 a.parquet
-rw------- 1 root root 2.8G Mar 30 01:46 Baby_Products.jsonl
-rw-r--r-- 1 root root  16K Mar 30 01:27 df_images_empty.parquet
-rw-r--r-- 1 root root 660M Mar 30 01:59 df_meta_2023.cleaned.in_review.jsonl
-rw-r--r-- 1 root root 661M Mar 30 01:28 df_meta_2023_cleaned.jsonl
-rw-r--r-- 1 root root 660M Mar 30 01:52 df_meta_2023_cleaned.v1.jsonl
-rw------- 1 root root 265M Mar 30 01:02 df_meta_2023.parquet
-rw-r--r-- 1 root root 264M Mar 30 01:37 df_meta_2023.v1.parquet
-rw-r--r-- 1 root root 1.3M Mar 30 01:55 df_reviews_not_in_meta.v1.jsonl
-rw-r--r-- 1 root root  62K Mar 30 01:26 df_title_null.parquet
drwx------ 5 root root 4.0K Mar 30 01:02 drive
-rw-r--r-- 1 root root    0 Mar 30 01:05 meta_2023_clean.parquet
-rw------- 1 root root 659M Mar 30 01:16 meta_Baby_Products.jsonl
drwxr-xr-x 1 root root 4.0K Mar 23 13:29 sample_data


In [4]:
df_meta_2023_cleaned = pd.read_json("df_meta_2023.cleaned.jsonl", lines=True)

In [5]:
# Đường dẫn file của bạn
file_path = "Baby_Products.jsonl"

# Cách sửa: Mở file trước rồi mới đọc
with open(file_path, 'r', encoding='utf-8') as f:
    review_chunks = pd.read_json(f, lines=True, chunksize=100_000)

    # Get unique parent_asin values from df_meta_2023 for efficient lookup
    meta_parent_asins = set(df_meta_2023_cleaned['parent_asin'].unique())

    # Initialize lists to store chunks
    all_reviews_in_meta_chunks = []
    all_reviews_not_in_meta_chunks = []

    print("Starting full chunk processing...")
    # Iterate through all chunks
    for i, chunk in enumerate(review_chunks):
        print(f"Processing chunk {i+1}...")
        # Create a boolean mask
        is_in_meta_mask = chunk['parent_asin'].isin(meta_parent_asins)

        # Append filtered chunks to respective lists
        all_reviews_in_meta_chunks.append(chunk[is_in_meta_mask].copy())
        all_reviews_not_in_meta_chunks.append(chunk[~is_in_meta_mask].copy())

    # Concatenate all collected chunks into final DataFrames
    df_reviews_in_meta = pd.concat(all_reviews_in_meta_chunks, ignore_index=True)
    df_reviews_not_in_meta = pd.concat(all_reviews_not_in_meta_chunks, ignore_index=True)

    print("\n--- First Part: Reviews Processing Complete ---")
    print("DataFrame with reviews whose parent_asin is in df_meta_2023:")
    display(df_reviews_in_meta.head())
    print(f"Shape of df_reviews_in_meta: {df_reviews_in_meta.shape}")

    print("\nDataFrame with reviews whose parent_asin is NOT in df_meta_2023:")
    display(df_reviews_not_in_meta.head())
    print(f"Shape of df_reviews_not_in_meta: {df_reviews_not_in_meta.shape}")

    # --- Second Part: Meta-data not in Reviews ---

    # Get all unique parent_asin values from the processed reviews (both in_meta and not_in_meta)
    all_reviews_parent_asins = pd.concat([
        df_reviews_in_meta['parent_asin'],
        df_reviews_not_in_meta['parent_asin']
    ]).unique()

    # Convert to a set for efficient lookup
    set_all_reviews_parent_asins = set(all_reviews_parent_asins)

    # Find parent_asin values in df_meta_2023 that are NOT in the reviews
    mask_meta_not_in_reviews = ~df_meta_2023_cleaned['parent_asin'].isin(set_all_reviews_parent_asins)
    df_meta_not_in_reviews = df_meta_2023_cleaned[mask_meta_not_in_reviews].copy()

    print("\n--- Second Part: Meta-data Processing Complete ---")
    print("Meta-data whose parent_asin is NOT in any of the processed reviews:")
    display(df_meta_not_in_reviews.head())
    print(f"Shape of df_meta_not_in_reviews: {df_meta_not_in_reviews.shape}")

Starting full chunk processing...
Processing chunk 1...
Processing chunk 2...
Processing chunk 3...
Processing chunk 4...
Processing chunk 5...
Processing chunk 6...
Processing chunk 7...
Processing chunk 8...
Processing chunk 9...
Processing chunk 10...
Processing chunk 11...
Processing chunk 12...
Processing chunk 13...
Processing chunk 14...
Processing chunk 15...
Processing chunk 16...
Processing chunk 17...
Processing chunk 18...
Processing chunk 19...
Processing chunk 20...
Processing chunk 21...
Processing chunk 22...
Processing chunk 23...
Processing chunk 24...
Processing chunk 25...
Processing chunk 26...
Processing chunk 27...
Processing chunk 28...
Processing chunk 29...
Processing chunk 30...
Processing chunk 31...
Processing chunk 32...
Processing chunk 33...
Processing chunk 34...
Processing chunk 35...
Processing chunk 36...
Processing chunk 37...
Processing chunk 38...
Processing chunk 39...
Processing chunk 40...
Processing chunk 41...
Processing chunk 42...
Processin

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,4,Good buy for preschool naps and home use...,I bought two of these for my kids for nap time...,[],B004FM7VOW,B089MS68G8,AGKASBHYZPGTEPO6LWZPVJWB2BVA,2016-08-18 18:52:17,1,True
1,5,THEY WORK- and are super cute to boot...,LOVE THESE! AND THEY WORK!!! I was on the fenc...,[],B01E5E703G,B01E5E703G,AGKASBHYZPGTEPO6LWZPVJWB2BVA,2016-08-18 17:44:04,1,True
2,1,cute but small and pretty much unusable as a c...,cute but small and pretty much unusable as a c...,[],B00F463XV8,B00F9386Q8,AGKASBHYZPGTEPO6LWZPVJWB2BVA,2016-01-13 02:08:01,0,True
3,5,Works great perfect size!,I have lots of different disposable diaper bag...,[],B0007V644S,B07RRDX26B,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,2014-08-25 19:14:11,0,True
4,5,Cute and Works Great,I was so excited for bath time when I register...,[],B002LARFLY,B00OLRJET6,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,2012-10-09 21:42:41,0,False


Shape of df_reviews_in_meta: (6025848, 10)

DataFrame with reviews whose parent_asin is NOT in df_meta_2023:


,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,4,3.5 stars--> Cuts watermelon well but other me...,I always like a good tool. This one works rela...,[{'small_image_url': 'https://images-na.ssl-im...,B07R12Z3JH,B07R12Z3JH,AFXF3EGQTQDXMRLDWFU7UBFQZB7Q,2019-06-26 00:48:41.833,0,False
1,5,It’s accurate,"I’m a nurse use it at work it’s fast, accurate...",[],B088DDXBDC,B088DDXBDC,AFUBGCA24SLWY6SQSNGFHQPXM5HA,2020-08-07 10:24:21.751,0,True
2,4,"Works, but messy",This slicer do-hicky works as advertised. But ...,[],B07R12Z3JH,B07R12Z3JH,AHE5RLELYXWSMUVORWWJI47PWU3A,2019-06-29 11:27:33.886,1,False
3,5,Absolutely love this carseat,This carseat is super easy to install. It seem...,[],B085HB69WR,B085HB69WR,AF6SM2ESLD534XWS7P6PFSL257CA,2021-01-19 16:42:39.111,0,False
4,5,Graco puts your mind at ease,This is a well designed safety oriented car se...,[],B07C2RNRCJ,B07C2RNRCJ,AEZG3K5Z4N75PPYQLZ7TYQVBQTHQ,2020-02-25 00:11:39.287,0,True


Shape of df_reviews_not_in_meta: (3036, 10)

--- Second Part: Meta-data Processing Complete ---
Meta-data whose parent_asin is NOT in any of the processed reviews:


,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
10379,Amazon Home,E-Coin Baby Prints Photo frame 1 Pack Newborn ...,4.7,25,[Baby print frame size: 13.8''Wx7.1''H x0.8''D...,[1],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Reviews from Real Moms', 'url': 'h...",XGATML,"[Baby Products, Nursery, Décor, Picture Frames]","{'Brand': 'TOMENGBEIAABBCC', 'Color': 'White',...",B09HKNZM2C,NaN,NaN,None
11714,Baby,Quinny Zapp 4 Travel System,3.5,2,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Quinny,"[Baby Products, Strollers & Accessories, Strol...",{},B001V8KTRO,NaN,NaN,None
17529,Baby,Vegan Leather Baby Play mat|Baby Play mats for...,4.5,55,[HIGH-QUALITY MATERIALS: Formaldehyde Free Cer...,[],129.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Gahroo round hazelnut/off white pl...,GAHROO,"[Baby Products, Activity & Entertainment, Baby...","{'Product Dimensions': '47 x 47 x 0.8 inches',...",B0BNGQFTCT,NaN,NaN,None
19714,All Beauty,(3 Pack) CareALL 2oz Zinc Oxide 20% Skin Prote...,4.6,1308,[SOOTHS AND PREVENTS diaper rash and chaffing....,[],5.99,[{'thumb': 'https://m.media-amazon.com/images/...,[],CareAll,"[Baby Products, Baby Care, Grooming, Skin Care...",{'Package Dimensions': '6.65 x 4.33 x 1.3 inch...,B09RJTLVJC,NaN,NaN,None
22976,Baby,New Mom Gift Set - Mama Mama Tumbler - Best Mo...,4.7,58,[★ PERFECT GIFT FOR EXPECTING Moms: It is ofte...,[],42.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Premium New Mom Gifts For Women wi...,Tipit Drinkware,"[Baby Products, Gifts, Gift Sets]","{'Product Dimensions': '11 x 10 x 4 inches', '...",B093CGBWXY,NaN,NaN,None


Shape of df_meta_not_in_reviews: (70, 16)


In [10]:
df_meta_not_in_reviews.info()

<class 'pandas.core.frame.DataFrame'>
Index: 70 entries, 10379 to 207264
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   main_category    66 non-null     object 
 1   title            70 non-null     object 
 2   average_rating   70 non-null     float64
 3   rating_number    70 non-null     int64  
 4   features         70 non-null     object 
 5   description      70 non-null     object 
 6   price            18 non-null     float64
 7   images           70 non-null     object 
 8   videos           70 non-null     object 
 9   store            70 non-null     object 
 10  categories       70 non-null     object 
 11  details          70 non-null     object 
 12  parent_asin      70 non-null     object 
 13  bought_together  0 non-null      float64
 14  subtitle         0 non-null      float64
 15  author           0 non-null      object 
dtypes: float64(4), int64(1), object(11)
memory usage: 9.3+ KB


In [14]:
df_meta_2023_cleaned[df_meta_2023_cleaned["parent_asin"].isnull()]

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author


In [26]:
df_meta_2023_cleaned_in_review = df_meta_2023[~(df_meta_2023["parent_asin"].isin(df_meta_not_in_reviews["parent_asin"]))]

In [27]:
df_meta_2023_cleaned_in_review.info()

<class 'pandas.core.frame.DataFrame'>
Index: 217610 entries, 0 to 217679
Data columns (total 16 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   main_category    199736 non-null  object 
 1   title            217610 non-null  object 
 2   average_rating   217610 non-null  float64
 3   rating_number    217610 non-null  int64  
 4   features         217610 non-null  object 
 5   description      217610 non-null  object 
 6   price            60234 non-null   float64
 7   images           217610 non-null  object 
 8   videos           217610 non-null  object 
 9   store            214608 non-null  object 
 10  categories       217610 non-null  object 
 11  details          217610 non-null  object 
 12  parent_asin      217610 non-null  object 
 13  bought_together  0 non-null       float64
 14  subtitle         0 non-null       float64
 15  author           1 non-null       object 
dtypes: float64(4), int64(1), object(11)
memory 

In [28]:
df_meta_2023_cleaned_in_review.to_json('df_meta_2023.cleaned.in_review.jsonl', orient='records', lines=True)

In [29]:
!cp "df_meta_2023.cleaned.in_review.jsonl" "{PATH}/2023/df_meta_2023.cleaned.in_review.jsonl"

In [30]:
df_reviews_in_meta.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6025848 entries, 0 to 6025847
Data columns (total 10 columns):
 #   Column             Dtype         
---  ------             -----         
 0   rating             int64         
 1   title              object        
 2   text               object        
 3   images             object        
 4   asin               object        
 5   parent_asin        object        
 6   user_id            object        
 7   timestamp          datetime64[ns]
 8   helpful_vote       int64         
 9   verified_purchase  bool          
dtypes: bool(1), datetime64[ns](1), int64(2), object(6)
memory usage: 419.5+ MB


In [8]:
df_reviews_in_meta.shape

(6025848, 10)

In [9]:
df_reviews_not_in_meta.shape

(3036, 10)

In [ ]:
# df_reviews_in_meta.to_json('df_review_2023.in_meta.jsonl', orient='records', lines=True)

In [6]:
output_file = 'df_review_2023.in_meta.jsonl'
chunk_size = 50000  # Chia nhỏ mỗi lần ghi 50k dòng

# Xóa file cũ nếu tồn tại
if os.path.exists(output_file):
    os.remove(output_file)

# Ghi từng cụm
for i in range(0, len(df_reviews_in_meta), chunk_size):
    chunk = df_reviews_in_meta.iloc[i : i + chunk_size]
    chunk.to_json(output_file, orient='records', lines=True, mode='a')
    # mode='a' là append (nối đuôi)

print("✅ Đã ghi xong bằng phương pháp Chunking!")

✅ Đã ghi xong bằng phương pháp Chunking!


In [7]:
!cp "df_review_2023.in_meta.jsonl" "{PATH}/2023/df_review_2023.in_meta.jsonl"